# Retail Sales Forecast Dataset 2026

This notebook performs EDA and builds baseline machine learning models for sales forecasting and demand classification.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report

## Find Dataset File

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Load Dataset

In [ ]:
csv_path = None

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'retail_sales_forecast_dataset.csv':
            csv_path = os.path.join(dirname, filename)

print('Dataset path:', csv_path)

df = pd.read_csv(csv_path)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

## Monthly Sales Trend

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
monthly_sales = df.groupby(df['Date'].dt.to_period('M'))['Sales_Amount'].sum()
monthly_sales.index = monthly_sales.index.astype(str)
monthly_sales.plot(figsize=(12, 5))
plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Total Sales Amount')
plt.xticks(rotation=45)
plt.show()

## Sales by Product Category

In [ ]:
df.groupby('Product_Category')['Sales_Amount'].sum().sort_values().plot(kind='bar')
plt.title('Total Sales by Product Category')
plt.xlabel('Product Category')
plt.ylabel('Total Sales Amount')
plt.show()

## Promotion Impact

In [ ]:
df.groupby('Promotion')['Sales_Amount'].mean().plot(kind='bar')
plt.title('Average Sales Amount by Promotion')
plt.xlabel('Promotion')
plt.ylabel('Average Sales Amount')
plt.show()

## Demand Level Distribution

In [ ]:
df['Demand_Level'].value_counts().plot(kind='bar')
plt.title('Demand Level Distribution')
plt.xlabel('Demand Level')
plt.ylabel('Count')
plt.show()

## Regression Model: Predict Sales Amount

In [ ]:
X = df.drop(columns=['Transaction_ID', 'Date', 'Sales_Amount', 'Demand_Level'])
y = df['Sales_Amount']

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(exclude=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

reg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
reg.fit(X_train, y_train)
pred = reg.predict(X_test)

print('MAE:', mean_absolute_error(y_test, pred))
print('R2 Score:', r2_score(y_test, pred))

## Classification Model: Predict Demand Level

In [ ]:
X = df.drop(columns=['Transaction_ID', 'Date', 'Sales_Amount', 'Demand_Level'])
y = df['Demand_Level']

clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print('Accuracy:', accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

## Results and Conclusion

This analysis explored the Retail Sales Forecast Dataset 2026 and identified important business factors influencing sales performance.

Key findings include:

- Promotions increased average sales performance.
- Seasonal and holiday effects influenced customer demand.
- Product category, customer footfall, marketing spend, and inventory levels contributed to sales variation.
- Baseline machine learning models were able to predict sales amount and classify demand levels using retail features.

This dataset is suitable for sales forecasting, demand prediction, regression, classification, retail analytics, and business intelligence projects.